In [ ]:
import numpy as np, pickle, sys, matplotlib as mpl, pickle, os
import matplotlib.pyplot as plt, matplotlib.colors as colors
from matplotlib import ticker, cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

if not os.environ.get("HOME_KADATH"):
    raise EnvironmentError("Please set the environment variable HOME_KADATH to the path of your FUKA installation.")
FUKA_path = str(os.environ["HOME_KADATH"])

pyFUKA_path = FUKA_path+'/codes/PythonTools'
pyFUKA_libspath = pyFUKA_path+'/lib/'
sys.path.append(pyFUKA_libspath)
%env HOME_KADATH=$FUKA_path

In [ ]:
# import custom FUKA modules
from fuka_plot_tools.setup_utils import get_reader

In [ ]:
initial_data_path=f"{pyFUKA_path}/Example_id/NS_ISO_DIFF_ROT.dd2.keh.madm.2.2.Ar.1.Rr.0.25.0.0565925.0.13.dat"
if os.path.isfile(initial_data_path) == True:
    readerISO = get_reader(initial_data_path, ns=False, ns_iso_diffrot=True, ns_iso_uniformrot=False)
else:
    print("{} not found",initial_data_path)

In [ ]:
# Here we can see the initial data solution variables that can be accessed
# Variables with type KadathScalar can be interpolated as shown below.
readerISO.vars

In [ ]:
# We can also access data directly from the INFO file
rr=readerISO.config['ns']['differential_rotation']['R_ratio']
print(f"{rr:.2f}")

In [ ]:
# Here we can setup the grid to interpolate data to
y1, y2 = [-20, 20]
npts = 128

y_coords = np.linspace(y1, y2, num=npts)

x1, x2 = [-20,20]
x_coords = np.linspace(x1, x2, num=npts)

coords_lst = [[x,y] for y in y_coords for x in x_coords]
X, Y = np.meshgrid(x_coords, y_coords)

In [ ]:
# Interpolated violates of the constraint on Omega
dataOME = readerISO.getFieldValues("comega", coords_lst, -1)

In [ ]:
dataome = np.array(dataOME)

In [ ]:
dataome=dataome.reshape(len(x_coords),len(x_coords))

In [ ]:
cs = plt.pcolor(X,Y,np.log10(np.abs(dataome)))
cbar = plt.colorbar(
        cs,ticklocation='right',
        orientation='vertical', extend='both',
        label=r"$\log_{10} \mathcal{H}$",
        #ticks=np.linspace(norm.vmin, norm.vmax, num=5)
      )

In [ ]:
# Now we can access the data that is exported to an evolution framework
# E.g. gamma_ij, K_ij, beta^i, lapse, U^i, and fluid quantities
for k in readerISO.getExporterKeys():
    print(k)

readerISO.vars

In [ ]:
# We restrict ourselves to the x-y plane, but this can
# easily be changed to the x-z or y-z plane
coords_lst_exp = [[x,y,0] for y in y_coords for x in x_coords]

In [ ]:
# Interpolate the y component of the fluid 3 velocity, U^2
interp_var= "vel2"
export_data_init = readerISO.getExporterFieldValues__cartesian(interp_var, coords_lst_exp)

In [ ]:
export_data = np.array(export_data_init)

In [ ]:
export_data=export_data.reshape(len(x_coords),len(x_coords))

In [ ]:
norm = None #colors.Normalize(vmin=-14, vmax=-3, clip=False)
fig, axs = plt.subplots(1,1)
cs = axs.pcolor(X, Y, export_data, norm=norm)
cbarax,kw = mpl.colorbar.make_axes(axs, location='right', orientation='vertical', aspect=30)
cbar = fig.colorbar(
        cs,
        cax=cbarax, ticklocation='right',
        orientation='vertical', extend='both',
        #ticks=np.linspace(norm.vmin, norm.vmax, num=5)
      )